(a

In [42]:
from data_reading import data, header
import pandas as pd

def calculate_mean(values):
    return sum(values) / len(values)

def calculate_std(values):
    mean = calculate_mean(values)
    squared_deviations = [(x - mean) ** 2 for x in values]
    variance = sum(squared_deviations) / (len(values) - 1)
    return variance ** 0.5

portfolio_names = header[4:]
results=[]

for portfolio_index in range(12):
    returns = [row["returns"][portfolio_index]
               for row in data]

    mean = calculate_mean(returns)
    std = calculate_std(returns)

    results.append({"Portfolio": portfolio_names[portfolio_index],"Mean": mean,"Std. Dev.": std})
results_table = pd.DataFrame(results)

print(results_table.to_string(index=False,
            formatters={
            "Mean": lambda value: f"{value:.4f}%",
            "Std. Dev.": lambda value: f"{value:.4f}%"
        }))

Portfolio    Mean Std. Dev.
    1-Dec 0.0511%   1.2798%
    2-Dec 0.0494%   1.3155%
    3-Dec 0.0505%   1.2423%
    4-Dec 0.0499%   1.2059%
    5-Dec 0.0484%   1.1887%
    6-Dec 0.0496%   1.1465%
    7-Dec 0.0469%   1.1452%
    8-Dec 0.0466%   1.1206%
    9-Dec 0.0441%   1.1090%
   10-Dec 0.0405%   1.0705%
   vwretd 0.0413%   1.0563%
   ewretd 0.0828%   1.0411%


(b

In [40]:
results_table["Monthly Mean"] = results_table["Mean"] * 21
results_table["Monthly Std. Dev."] = results_table["Std. Dev."] * (21 ** 0.5)
results_table["Annual Mean"] = results_table["Mean"] * 252
results_table["Annual Std. Dev."] = results_table["Std. Dev."] * (252 ** 0.5)

print(results_table.to_string(index=False,
        formatters={
            column: lambda value: f"{value:.4f}%"
            for column in results_table.columns
            if column != "Portfolio"}))

Portfolio    Mean Std. Dev. Monthly Mean Monthly Std. Dev. Annual Mean Annual Std. Dev.
    1-Dec 0.0511%   1.2798%      1.0731%           5.8646%    12.8774%         20.3157%
    2-Dec 0.0494%   1.3155%      1.0371%           6.0286%    12.4456%         20.8836%
    3-Dec 0.0505%   1.2423%      1.0614%           5.6927%    12.7365%         19.7202%
    4-Dec 0.0499%   1.2059%      1.0470%           5.5262%    12.5642%         19.1432%
    5-Dec 0.0484%   1.1887%      1.0171%           5.4475%    12.2053%         18.8708%
    6-Dec 0.0496%   1.1465%      1.0407%           5.2540%    12.4889%         18.2003%
    7-Dec 0.0469%   1.1452%      0.9859%           5.2481%    11.8311%         18.1800%
    8-Dec 0.0466%   1.1206%      0.9796%           5.1353%    11.7546%         17.7892%
    9-Dec 0.0441%   1.1090%      0.9253%           5.0821%    11.1041%         17.6049%
   10-Dec 0.0405%   1.0705%      0.8497%           4.9054%    10.1963%         16.9929%
   vwretd 0.0413%   1.0563%     

(c
Yes, we could use an OLS regression with only a constant term :

In [43]:
def estimate_intercept_ols(values):
    x_values = [1.0 for _ in values]

    x_squared_sum = sum(x * x for x in x_values)
    x_y_sum = sum(x * y for x, y in zip(x_values, values))

    return x_y_sum / x_squared_sum

regression_results = []
for portfolio_index in range(12):
    returns = [
        row["returns"][portfolio_index]
        for row in data]

    direct_mean = calculate_mean(returns)
    ols_intercept = estimate_intercept_ols(returns)

    regression_results.append({
        "Portfolio": portfolio_names[portfolio_index],
        "Direct Mean (%)": f"{direct_mean:.4f}%",
        "OLS Intercept (%)": f"{ols_intercept:.4f}%"
    })


regression_table = pd.DataFrame(regression_results)

print(regression_table.to_string(index=False))

Portfolio Direct Mean (%) OLS Intercept (%)
    1-Dec         0.0511%           0.0511%
    2-Dec         0.0494%           0.0494%
    3-Dec         0.0505%           0.0505%
    4-Dec         0.0499%           0.0499%
    5-Dec         0.0484%           0.0484%
    6-Dec         0.0496%           0.0496%
    7-Dec         0.0469%           0.0469%
    8-Dec         0.0466%           0.0466%
    9-Dec         0.0441%           0.0441%
   10-Dec         0.0405%           0.0405%
   vwretd         0.0413%           0.0413%
   ewretd         0.0828%           0.0828%


(d

In [44]:
cutoff_date = 19450101
d_results = []

for i in range(12):
    pre = [row["returns"][i] for row in data if int(row["date"]) < cutoff_date]
    post = [row["returns"][i] for row in data if int(row["date"]) >= cutoff_date]

    pre_mean = calculate_mean(pre)
    post_mean = calculate_mean(post)

    pre_std = calculate_std(pre)
    post_std = calculate_std(post)

    standard_error = (pre_std**2 / len(pre) + post_std**2 / len(post)) ** 0.5

    t_stat = (pre_mean - post_mean) / standard_error

    d_results.append({
        "Portfolio": portfolio_names[i],
        "Pre-1945 Mean": f"{pre_mean:.4f}",
        "Post-1945 Mean": f"{post_mean:.4f}",
        "t-statistic": f"{t_stat:.4f}",
        "Reject H0": "Yes" if abs(t_stat) > 1.96 else "No"
    })

d_table = pd.DataFrame(d_results)

print(d_table.to_string(index=False))

Portfolio Pre-1945 Mean Post-1945 Mean t-statistic Reject H0
    1-Dec        0.0572         0.0493      0.2666        No
    2-Dec        0.0467         0.0502     -0.1271        No
    3-Dec        0.0409         0.0533     -0.4982        No
    4-Dec        0.0434         0.0517     -0.3437        No
    5-Dec        0.0364         0.0519     -0.6605        No
    6-Dec        0.0441         0.0511     -0.3065        No
    7-Dec        0.0306         0.0517     -0.9214        No
    8-Dec        0.0325         0.0507     -0.8270        No
    9-Dec        0.0282         0.0487     -0.9254        No
   10-Dec        0.0263         0.0446     -0.9009        No
   vwretd        0.0275         0.0453     -0.8674        No
   ewretd        0.1259         0.0704      2.4950       Yes
